In [7]:
from astropy.table import Table, vstack, join, setdiff
import glob
import numpy as np

from astropy.coordinates import SkyCoord
from astropy import units as u

import matplotlib.pyplot as plt

#from astropy.io import fits

from tqdm import tqdm

import glob
import os

In [6]:
tf_loa=Table.read('/global/cfs/projectdirs/desi/science/td/pv/tfgalaxies/Y3/desi_pv_loa_healpix.fits')
tf_loa[:5]

TARGETID,TARGET_RA,TARGET_DEC,Z,ZERR,SPECTYPE,DELTACHI2,ZWARN,PVTYPE,SGA_ID,PHOTSYS
int64,float64,float64,float64,float64,bytes6,float64,int64,bytes3,int64,bytes1
-430502046,134.00013295228783,5.934552839997555,1.5211812148930146,8.936834292063904e-05,GALAXY,22.838147282600403,2049,TFT,838970,S
-427872363,61.981718642816155,-22.823913110732356,0.049086072011651605,8.769259809810332e-06,GALAXY,320.07210237160325,2049,TFT,982213,S
-411444222,156.2181500220905,7.1584828480845655,0.7655255485455215,0.0001486141274558249,GALAXY,3.784611649578437,2053,TFT,4614,S
-261707523,51.18809619857989,-15.380001206793725,0.11707091119461288,1.2658057107331122e-05,GALAXY,5004.164042882621,512,SGA,788458,S
-260779407,138.32747528464944,17.5639019859795,0.08569268559472955,9.449907878073889e-06,GALAXY,237.52352766692638,2560,TFT,735997,S


In [32]:
tf_iron = Table.read('/global/cfs/cdirs/desi/science/td/pv/tfgalaxies/desi_pv_tf_iron_healpix.fits')
tf_iron[:5]

TARGETID,TARGET_RA,TARGET_DEC,HEALPIX,SURVEY,Z,ZERR,ZWARN,DELTACHI2,FILENAME,PVTYPE,SGA_ID,RA,DEC
int64,float64,float64,int64,bytes4,float64,float64,int64,float64,bytes65,bytes3,int64,float64,float64
2852147603439621,198.369130660983,36.5372037049171,10475,main,0.815976335547845,7.38513168100107e-05,4,0.128754377365112,iron/healpix/main/dark/104/10475/redrock-main-dark-10475.fits,EXT,649377,198.36913066098333,36.537203704917076
2399148812795907,198.371733180003,36.4994335406917,10475,main,1.11088784970434,7.48767797671894e-05,4,7.9473560154438,iron/healpix/main/bright/104/10475/redrock-main-bright-10475.fits,EXT,649377,198.37173318000336,36.499433540691676
2399382443917318,184.845242475328,49.8157304793777,10995,main,1.14739342108157,0.000146302276719084,4,2.56771463155746,iron/healpix/main/bright/109/10995/redrock-main-bright-10995.fits,EXT,1008911,184.84524247532795,49.81573047937771
2399634072797192,184.341289722203,70.8283725474297,11965,main,1.51703376230705,6.28979649962091e-05,4,4.76254060305655,iron/healpix/main/bright/119/11965/redrock-main-bright-11965.fits,EXT,241234,184.34128972220284,70.82837254742968
2852141710442505,123.256011148025,36.2652948002806,6448,main,0.00787379494184006,3.4714052819995e-05,0,22.1719104201402,iron/healpix/main/dark/64/6448/redrock-main-dark-6448.fits,EXT,31591,123.25601114802525,36.26529480028061


In [33]:
tf_fuji = Table.read('/global/cfs/cdirs/desi/science/td/pv/tfgalaxies/desi_pv_tf_fuji_healpix.fits')
tf_fuji[:5]

TARGETID,TARGET_RA,TARGET_DEC,HEALPIX,SURVEY,Z,ZERR,ZWARN,DELTACHI2,FILENAME,PVTYPE,SGA_ID,RA,DEC
int64,float64,float64,int64,bytes3,float64,float64,int64,float64,bytes63,bytes3,int64,float64,float64
1079550234591232,194.390863195343,27.5157211790145,10378,sv3,1.1235686466514,7.31685779475115e-05,4,3.28414569795132,fuji/healpix/sv3/bright/103/10378/redrock-sv3-bright-10378.fits,EXT,662902,194.39086319534337,27.51572117901454
1092744374124544,194.390863195343,27.5157211790145,10378,sv3,0.686773088332363,6.9756676262104e-05,4,0.786607094109058,fuji/healpix/sv3/dark/103/10378/redrock-sv3-dark-10378.fits,EXT,662902,194.39086319534337,27.51572117901454
1092744374124546,194.364461113654,27.5037185881314,10378,sv3,0.0242933923052181,4.95233472646785e-05,0,95.428411073226,fuji/healpix/sv3/dark/103/10378/redrock-sv3-dark-10378.fits,EXT,662902,194.36446111365385,27.50371858813136
1092744369930240,194.338458724402,27.4918902690326,10378,sv3,0.0264170223697961,0.00010139452689994,0,9.53278421035066,fuji/healpix/sv3/dark/103/10378/redrock-sv3-dark-10378.fits,EXT,662902,194.33845872440244,27.491890269032595
1092744374124545,194.377858465028,27.5098100780282,10378,sv3,0.211332646769145,6.68535116703737e-05,4,3.73989077657461,fuji/healpix/sv3/dark/103/10378/redrock-sv3-dark-10378.fits,EXT,662902,194.3778584650283,27.509810078028195


In [5]:
shredded_list = list(glob.glob('/pscratch/sd/n/nravi/fastspecfit_shredded/*.fits'))

shredded_tables = [Table.read(x) for x in shredded_list]

shredded = vstack(shredded_tables)

# shredded = vstack(shredded_tables, metadata_conflicts = 'silent')
shredded[:5]

TARGETID,SURVEY,PROGRAM,HEALPIX,TILEID_LIST,RA,DEC,COADD_FIBERSTATUS,CMX_TARGET,DESI_TARGET,BGS_TARGET,MWS_TARGET,SCND_TARGET,SV1_DESI_TARGET,SV1_BGS_TARGET,SV1_MWS_TARGET,SV2_DESI_TARGET,SV2_BGS_TARGET,SV2_MWS_TARGET,SV3_DESI_TARGET,SV3_BGS_TARGET,SV3_MWS_TARGET,SV1_SCND_TARGET,SV2_SCND_TARGET,SV3_SCND_TARGET,Z,ZWARN,DELTACHI2,SPECTYPE,SUBTYPE,Z_RR,ZWARN_RR,TSNR2_BGS,TSNR2_LRG,TSNR2_ELG,TSNR2_QSO,TSNR2_LYA,PHOTSYS,LS_ID,FIBERFLUX_G,FIBERFLUX_R,FIBERFLUX_Z,FIBERTOTFLUX_G,FIBERTOTFLUX_R,FIBERTOTFLUX_Z,FLUX_G,FLUX_R,FLUX_Z,FLUX_W1,FLUX_W2,FLUX_W3,FLUX_W4,FLUX_IVAR_G,FLUX_IVAR_R,FLUX_IVAR_Z,FLUX_IVAR_W1,FLUX_IVAR_W2,FLUX_IVAR_W3,FLUX_IVAR_W4,EBV,MW_TRANSMISSION_G,MW_TRANSMISSION_R,MW_TRANSMISSION_Z,MW_TRANSMISSION_W1,MW_TRANSMISSION_W2,MW_TRANSMISSION_W3,MW_TRANSMISSION_W4,SGA_ID
int64,bytes7,bytes6,int32,bytes365,float64,float64,int32,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,float64,int64,float64,bytes6,bytes20,float64,int64,float32,float32,float32,float32,float32,bytes1,int64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float64,float32,float32,float32,float32,float32,float32,float32,int64
1084427169955849,sv3,dark,10147,475,242.72318746350186,53.635782093163655,0,0,0,0,0,0,0,0,0,0,0,0,4611686018427387904,0,0,0,0,70368744177664,0.03063192091855999,0,789.0584922824055,GALAXY,--,0.03063192091855999,0,7225.891,85.85588,119.355064,30.441978,93.52414,S,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.014221867021468187,0.95879936,0.9720521,0.98426247,0.99759275,0.9985209,0.9996844,0.9998808,556089
1084434300272644,sv3,dark,10147,305,243.65642926326169,54.27724177905015,0,0,0,0,0,0,0,0,0,0,0,0,4611686018427387904,0,0,0,0,70368744177664,0.07850942434688926,0,310.9083716990426,GALAXY,--,0.07850942434688926,0,8733.865,105.43734,166.61946,42.03425,172.49423,S,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.009166928940794818,0.9732453,0.98189515,0.9898276,0.99844766,0.9990464,0.99979657,0.99992317,375898
1084427178344451,sv3,dark,10147,306,243.62537013307835,53.712014537285526,0,0,0,0,0,0,0,0,0,0,0,0,4611686018427387904,0,0,0,0,70368744177664,0.05723634298805001,0,849.3460895419121,GALAXY,--,0.05723634298805001,0,8027.5,95.33189,148.73509,37.593582,152.26321,S,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.010070448685360667,0.9706474,0.9801285,0.98883057,0.99829483,0.99895245,0.9997765,0.9999156,21266
1084434300272643,sv3,dark,10147,307,243.65118121249577,54.275090804072406,0,0,0,0,0,0,0,0,0,0,0,0,4611686018427387904,0,0,0,0,70368744177664,0.07974131508417623,0,1123.4490712519037,GALAXY,--,0.07974131508417623,0,7896.612,88.073654,147.83482,35.323612,96.00605,S,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.009137625924099202,0.9733297,0.9819525,0.9898599,0.99845266,0.9990494,0.99979717,0.9999234,375898
1084427178344450,sv3,dark,10147,308,243.62044993111272,53.713173370492754,0,0,0,0,0,0,0,0,0,0,0,0,4611686018427387904,0,0,0,0,70368744177664,0.05591772316967483,0,2463.962895900011,GALAXY,--,0.05591772316967483,0,8062.4043,92.058235,146.52583,35.603767,110.88087,S,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.010063057608963625,0.9706686,0.98014295,0.9888387,0.9982961,0.9989532,0.99977666,0.99991566,21266


In [42]:
shredded_loa = setdiff(tf_loa, shredded, keys = 'TARGETID')
shredded_fuji = setdiff(tf_fuji, shredded, keys = 'TARGETID')
shredded_iron = setdiff(tf_iron, shredded, keys = 'TARGETID')
shredded_loa_iron = setdiff(shredded_iron, shredded_loa, keys = 'TARGETID')

In [56]:
# take only the columns I need, and renaming them to match code already written

shredded_new = shredded[['TARGETID', 'SGA_ID', 'RA', 'DEC', 'Z', 'Z_RR', 'ZWARN', 'DELTACHI2']]
shredded_new.rename_column('RA', 'TARGET_RA')
shredded_new.rename_column('DEC', 'TARGET_DEC')
shredded_new.rename_column('Z_RR', 'Z_ERR')
shredded_new[:5]

TARGETID,SGA_ID,TARGET_RA,TARGET_DEC,Z,Z_ERR,ZWARN,DELTACHI2
int64,int64,float64,float64,float64,float64,int64,float64
1084427169955849,556089,242.72318746350186,53.635782093163655,0.03063192091855999,0.03063192091855999,0,789.0584922824055
1084434300272644,375898,243.65642926326169,54.27724177905015,0.07850942434688926,0.07850942434688926,0,310.9083716990426
1084427178344451,21266,243.62537013307835,53.712014537285526,0.05723634298805001,0.05723634298805001,0,849.3460895419121
1084434300272643,375898,243.65118121249577,54.275090804072406,0.07974131508417623,0.07974131508417623,0,1123.4490712519037
1084427178344450,21266,243.62044993111272,53.713173370492754,0.05591772316967483,0.05591772316967483,0,2463.962895900011


In [59]:
diff = vstack([shredded_loa, shredded_fuji, shredded_loa_iron])
diff[:5]

TARGETID,TARGET_RA,TARGET_DEC,Z,ZERR,SPECTYPE,DELTACHI2,ZWARN,PVTYPE,SGA_ID,PHOTSYS,HEALPIX,SURVEY,FILENAME,RA,DEC
int64,float64,float64,float64,float64,bytes6,float64,int64,bytes3,int64,bytes1,int64,bytes4,bytes65,float64,float64
-430502046,134.00013295228783,5.934552839997555,1.5211812148930146,8.936834292063904e-05,GALAXY,22.838147282600403,2049,TFT,838970,S,--,--,--,--,--
-427872363,61.981718642816155,-22.823913110732356,0.049086072011651605,8.769259809810332e-06,GALAXY,320.07210237160325,2049,TFT,982213,S,--,--,--,--,--
-411444222,156.2181500220905,7.1584828480845655,0.7655255485455215,0.0001486141274558249,GALAXY,3.784611649578437,2053,TFT,4614,S,--,--,--,--,--
-261707523,51.18809619857989,-15.380001206793725,0.11707091119461288,1.2658057107331122e-05,GALAXY,5004.164042882621,512,SGA,788458,S,--,--,--,--,--
-260779407,138.32747528464944,17.5639019859795,0.08569268559472955,9.449907878073889e-06,GALAXY,237.52352766692638,2560,TFT,735997,S,--,--,--,--,--


In [68]:
diff_new = diff[['TARGETID', 'SGA_ID', 'TARGET_RA', 'TARGET_DEC', 'Z', 'ZERR', 'ZWARN', 'DELTACHI2']]
diff_new.rename_column('ZERR', 'Z_ERR')
diff_new[:5]

TARGETID,SGA_ID,TARGET_RA,TARGET_DEC,Z,Z_ERR,ZWARN,DELTACHI2
int64,int64,float64,float64,float64,float64,int64,float64
-430502046,838970,134.00013295228783,5.934552839997555,1.5211812148930146,8.936834292063904e-05,2049,22.838147282600403
-427872363,982213,61.981718642816155,-22.823913110732356,0.049086072011651605,8.769259809810332e-06,2049,320.07210237160325
-411444222,4614,156.2181500220905,7.1584828480845655,0.7655255485455215,0.0001486141274558249,2053,3.784611649578437
-261707523,788458,51.18809619857989,-15.380001206793725,0.11707091119461288,1.2658057107331122e-05,512,5004.164042882621
-260779407,735997,138.32747528464944,17.5639019859795,0.08569268559472955,9.449907878073889e-06,2560,237.52352766692638


In [73]:
# take all the galaxies in shredded that have a missing value (in diff_new)
mask = np.isin(shredded_new['SGA_ID'], diff_new['SGA_ID'])
shredded_filtered = shredded_new[mask]
shredded_missing = vstack([shredded_filtered, diff_new])
shredded_missing

TARGETID,SGA_ID,TARGET_RA,TARGET_DEC,Z,Z_ERR,ZWARN,DELTACHI2
int64,int64,float64,float64,float64,float64,int64,float64
1084427169955849,556089,242.72318746350186,53.635782093163655,0.03063192091855999,0.03063192091855999,0,789.0584922824055
1084434300272644,375898,243.65642926326169,54.27724177905015,0.07850942434688926,0.07850942434688926,0,310.9083716990426
1084427178344451,21266,243.62537013307835,53.712014537285526,0.05723634298805001,0.05723634298805001,0,849.3460895419121
1084434300272643,375898,243.65118121249577,54.275090804072406,0.07974131508417623,0.07974131508417623,0,1123.4490712519037
1084427178344450,21266,243.62044993111272,53.713173370492754,0.05591772316967483,0.05591772316967483,0,2463.962895900011
1084423579631621,586736,243.5971911694678,53.60159524782599,1.501942887672439,1.501942887672439,4,5.1466483026742935
1084434300272642,375898,243.64847788500722,54.273982637133216,1.0466322203136944,1.0466322203136944,4,0.7738340049982071
1084437831876610,967351,244.18532954426558,54.52142727151777,0.029571317023910432,0.029571317023910432,0,1919.5805545002222
1084437836070915,254294,244.53285762867722,54.40391771756543,0.018319117301656092,0.018319117301656092,0,7107.103994041681


In [75]:
SGA = Table.read('/global/cfs/cdirs/cosmo/data/sga/2020/SGA-2020.fits', 'ELLIPSE')
SGA[:5]

SGA_ID,SGA_GALAXY,GALAXY,PGC,RA_LEDA,DEC_LEDA,MORPHTYPE,PA_LEDA,D25_LEDA,BA_LEDA,Z_LEDA,SB_D25_LEDA,MAG_LEDA,BYHAND,REF,GROUP_ID,GROUP_NAME,GROUP_MULT,GROUP_PRIMARY,GROUP_RA,GROUP_DEC,GROUP_DIAMETER,BRICKNAME,RA,DEC,D26,D26_REF,PA,BA,RA_MOMENT,DEC_MOMENT,SMA_MOMENT,G_SMA50,R_SMA50,Z_SMA50,SMA_SB22,SMA_SB22.5,SMA_SB23,SMA_SB23.5,SMA_SB24,SMA_SB24.5,SMA_SB25,SMA_SB25.5,SMA_SB26,G_MAG_SB22,R_MAG_SB22,Z_MAG_SB22,G_MAG_SB22.5,R_MAG_SB22.5,Z_MAG_SB22.5,G_MAG_SB23,R_MAG_SB23,Z_MAG_SB23,G_MAG_SB23.5,R_MAG_SB23.5,Z_MAG_SB23.5,G_MAG_SB24,R_MAG_SB24,Z_MAG_SB24,G_MAG_SB24.5,R_MAG_SB24.5,Z_MAG_SB24.5,G_MAG_SB25,R_MAG_SB25,Z_MAG_SB25,G_MAG_SB25.5,R_MAG_SB25.5,Z_MAG_SB25.5,G_MAG_SB26,R_MAG_SB26,Z_MAG_SB26,SMA_SB22_ERR,SMA_SB22.5_ERR,SMA_SB23_ERR,SMA_SB23.5_ERR,SMA_SB24_ERR,SMA_SB24.5_ERR,SMA_SB25_ERR,SMA_SB25.5_ERR,SMA_SB26_ERR,G_MAG_SB22_ERR,R_MAG_SB22_ERR,Z_MAG_SB22_ERR,G_MAG_SB22.5_ERR,R_MAG_SB22.5_ERR,Z_MAG_SB22.5_ERR,G_MAG_SB23_ERR,R_MAG_SB23_ERR,Z_MAG_SB23_ERR,G_MAG_SB23.5_ERR,R_MAG_SB23.5_ERR,Z_MAG_SB23.5_ERR,G_MAG_SB24_ERR,R_MAG_SB24_ERR,Z_MAG_SB24_ERR,G_MAG_SB24.5_ERR,R_MAG_SB24.5_ERR,Z_MAG_SB24.5_ERR,G_MAG_SB25_ERR,R_MAG_SB25_ERR,Z_MAG_SB25_ERR,G_MAG_SB25.5_ERR,R_MAG_SB25.5_ERR,Z_MAG_SB25.5_ERR,G_MAG_SB26_ERR,R_MAG_SB26_ERR,Z_MAG_SB26_ERR,G_COG_PARAMS_MTOT,G_COG_PARAMS_M0,G_COG_PARAMS_ALPHA1,G_COG_PARAMS_ALPHA2,G_COG_PARAMS_CHI2,R_COG_PARAMS_MTOT,R_COG_PARAMS_M0,R_COG_PARAMS_ALPHA1,R_COG_PARAMS_ALPHA2,R_COG_PARAMS_CHI2,Z_COG_PARAMS_MTOT,Z_COG_PARAMS_M0,Z_COG_PARAMS_ALPHA1,Z_COG_PARAMS_ALPHA2,Z_COG_PARAMS_CHI2,ELLIPSEBIT
int64,bytes16,bytes29,int64,float64,float64,bytes21,float32,float32,float32,float32,float32,float32,bool,bytes13,int64,bytes35,int16,bool,float64,float64,float32,bytes8,float64,float64,float32,bytes4,float32,float32,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int32
2,SGA-2020 2,PGC1283207,1283207,228.3770865,5.4232017,S?,152.2,0.36307806,0.724436,0.03463229,23.40448,16.976,False,LEDA-20181114,0,PGC1283207,1,True,228.3770865,5.4232017,0.36307806,2283p055,228.3770803831908,5.423191398593787,0.49470574,SB26,158.20142,0.545691,228.37700918822188,5.4232652570544015,10.897086,3.3509698,3.1147978,3.240862,5.902337,6.9126143,7.941369,8.997992,10.073601,11.199986,12.391357,13.561038,14.841172,16.966799,16.108246,15.486356,16.879545,16.024958,15.400715,16.818878,15.967034,15.341793,16.776297,15.925804,15.300776,16.746685,15.897334,15.272053,16.725166,15.876816,15.2521105,16.708357,15.862035,15.237181,16.696539,15.851936,15.226998,16.689613,15.844313,15.21976,0.013392451,0.02354,0.021872982,0.01736985,0.024445537,0.039866067,0.05026544,0.08455789,0.122911856,0.005682776,0.0054258136,0.0049038026,0.005588406,0.005323561,0.0047632363,0.00543534,0.005177031,0.0046343105,0.0053025587,0.005040888,0.0045181247,0.005206092,0.0049438984,0.0044374703,0.0051483097,0.0048758644,0.0043834248,0.0051032505,0.0048264163,0.004344248,0.0050705094,0.004792021,0.004319857,0.005054293,0.004765629,0.0043044444,16.65942,0.34037337,0.2978292,3.0239506,0.07928849,15.820566,0.2640441,0.34559453,3.3033552,0.003811298,15.195567,0.29826432,0.3001073,3.2333765,0.011723555,0
3,SGA-2020 3,PGC1310416,1310416,202.54443750000002,6.9345944,Sc,159.26,0.4017908,0.7816278,0.073888786,23.498482,16.85,False,LEDA-20181114,1,PGC1310416,1,True,202.54443750000002,6.9345944,0.4017908,

In [76]:
SGA_dict = {}
for i in range(len(SGA)):
    SGA_dict[SGA['SGA_ID'][i]] = i

In [77]:
tf_loa = shredded_missing

In [78]:
#creates empty columns to add to tf_loa for classifications
# projected distance between center and fiber
tf_loa['DIST'] = np.nan

# normalized distance between center and fiber
tf_loa['DIST_R26'] = np.nan

# pa of major axis
tf_loa['PA'] = np.nan

# position angle between center point and fiber
tf_loa['C_TO_F_ANGLE'] = np.nan

# position angle of fiber and major axis
tf_loa['ANGLE_OFF_AXIS'] = np.nan

for i in tqdm(np.unique(tf_loa['SGA_ID'])):

    #identify all galaxy targets
    targ_id = tf_loa['SGA_ID'] == i

    #find the index for this target in SGA
    sga_id = SGA_dict[i]
    
#------------------------------------------
#find the coordinates of center of galaxy
#------------------------------------------
    SGA_coords = SkyCoord(ra=SGA['RA'][sga_id], 
                          dec=SGA['DEC'][sga_id], 
                          unit=u.degree)
    
#---------------------------------------------------   
#find the coordinates of all the targets on galaxy
#---------------------------------------------------  
    target_coords = SkyCoord(ra=tf_loa['TARGET_RA'][targ_id], 
                             dec=tf_loa['TARGET_DEC'][targ_id], 
                             unit=u.degree)
    
    # separation between center and fiber (in deg)
    sep2d = target_coords.separation(SGA_coords)
    
#---------------------------------
# find if fiber is on major axis
#---------------------------------
    
    # position angle of semimajor axis
    pa = SGA['PA'][sga_id]*u.deg

    # position angle between center and fiber
    theta = SGA_coords.position_angle(target_coords).to(u.deg).value

    # angle between semimajor axis and fiber
    sep_angle = theta - pa.value

    # replace all points in center with 0
    sep_angle = abs(np.where(theta == 0, 0, sep_angle))

    # replace all angles close to 180 or 0 with 0 --------------------------------------------
    tol = (np.isclose(sep_angle, 0, atol = 1e-4)) | (np.isclose(sep_angle, 180, atol = 1e-4))

    sep_angle[tol] = 0
    #-------------------------------------------------------------------------------------------

    #print('pa: ', pa.value,'theta: ', theta,'sep: ', sep_angle)

    # distance to center of galaxy
    tf_loa['DIST'][targ_id] = sep2d
    
    # distance to target coordinates (in units R26)
    tf_loa['DIST_R26'][targ_id] = 2*sep2d.to('arcmin')/(SGA['D26'][sga_id]*u.arcmin)

    tf_loa['C_TO_F_ANGLE'][targ_id] = theta
    
    tf_loa['ANGLE_OFF_AXIS'][targ_id] = sep_angle

    tf_loa['PA'][targ_id] = pa

100%|██████████| 2613/2613 [00:03<00:00, 662.52it/s]


In [79]:
#target distance that can be considered in center
#units R26
center_dist_lim = 0.001

#minimum distance between two targets to be considered unique
#units R26
unique_dist_lim = 0.01

In [80]:
#empty column to classify galaxy
tf_loa['Selection'] = 0

#for each unique SGA ID
for i in tqdm(np.unique(tf_loa['SGA_ID'])):

    #identify all the galaxies and targets
    obs_id = np.logical_and(tf_loa['SGA_ID'] == i, tf_loa['TARGETID'] > 0)
    #logical_and returns if both statements are true
    
    #makes a table of the targets corresponding to this galaxy
    obs = tf_loa[obs_id]

    sga_id = SGA_dict[i] 

#if the galaxy has more than three observations
    if len(obs) >= 3:

        # identify all points on semimajor axis
        on_major = obs[obs['ANGLE_OFF_AXIS'] == 0]

        # find number of unique points on axis
        if len(on_major) == 0:
            unique_on = 0

        else:
            on_major['DIST_R26'] = np.round(on_major['DIST_R26'], 6)
            on_major = on_major.group_by('DIST_R26')
            unique_on = len(on_major.groups)
        
        # identify all points off major axis
        off_major = obs[obs['ANGLE_OFF_AXIS'] != 0]

        # find number of unique points off axis
        num_off_major = len(off_major)

        if num_off_major == 0:
            unique_off = 0
        else:
            off_major = off_major.group_by(['DIST_R26','ANGLE_OFF_AXIS'])
            unique_off = len(off_major.groups)

#-----
# a
#-----
        #check to see if there is a center observation
        if np.any(on_major['DIST_R26'] <= center_dist_lim):

            # check how many unique points on semimajor axis
            not_center = on_major[on_major['DIST_R26'] > unique_dist_lim]

            if len(not_center) >= 2 and (np.max(not_center['DIST_R26']) - np.min(not_center['DIST_R26'])) >= unique_dist_lim:
                
                #since there 2 unique points, classify it as viable galaxy
                tf_loa['Selection'][obs_id] = 1

            # check how many points off semimajor axis
            elif (len(not_center) >= 1) and (num_off_major >= 1):
                tf_loa['Selection'][obs_id] = 1

            elif num_off_major >= 2:
                tf_loa['Selection'][obs_id] = 1

#----
# b
#----

        elif unique_on + unique_off >= 10:
            tf_loa['Selection'][obs_id] = 1
            # print(sga_id,': on axis:', on_major['DIST_R26'], len(on_major.groups))
            # print(sga_id,': off axis:', off_major['DIST_R26'], off_major['ANGLE_OFF_AXIS'], unique_off)
#----
# c
#----
        #since there isn't a center observation
        elif len(on_major) > 0:
            
            #split targets on semimajor axis onto either side of center galaxy
            if (SGA['PA'][sga_id] < 45) or (SGA['PA'][sga_id] > 135):

                left_index = on_major['TARGET_DEC'] - SGA['DEC'][sga_id] > 0

            else:
                left_index = on_major['TARGET_RA'] - SGA['RA'][sga_id] > 0
                
            left = on_major[left_index]
            right = on_major[~left_index]

            if len(left) > 0 and len(right) > 0:

                for j in range(len(left)):
                    
                    # check that there are 2 symmetric observations
                    if np.any(np.abs(right['DIST_R26'] - left['DIST_R26'][j]) < unique_dist_lim):

                    #check if there is a third point
                        if (np.any(np.abs(right['DIST_R26'] - left['DIST_R26'][j]) > unique_dist_lim) 
                            or np.any(np.abs(left['DIST_R26'] - left['DIST_R26'][j]) > unique_dist_lim)
                            or (num_off_major != 0)):

                            #viable galaxy
                            tf_loa['Selection'][obs_id] = 1
                            
    # if tf_loa['Selection'][obs_id][0] == 1:
    #     print(obs['SGA_ID'][0])

print(len(tf_loa[tf_loa['Selection']==1]))

100%|██████████| 2613/2613 [00:05<00:00, 508.42it/s]

17630


In [81]:
print(len(tf_loa[tf_loa['Selection']==1]))
loa_targs = (tf_loa[tf_loa['Selection']==1])
len(np.unique(loa_targs['SGA_ID']))

17630


1277

In [82]:
loa_targs.write('/pscratch/sd/d/dbustos/rot_curves/loa_targs_missing.fits',format='fits',overwrite=True)